In [ ]:
import pandas as pd 

df = pd.read_csv('drug_design.csv')
df.shape # 약 160만개의 데이터

(1576904, 2)

In [ ]:
# smiles가 중복되는 것이 존재하면 삭제함
df = df.drop_duplicates('SMILES')
df = df.reset_index(drop=True)
df.shape

(1503672, 3)

In [ ]:
import numpy as np

data = [len(i) for i in df['SMILES']]

q1 = np.percentile(data, 25)  # 1사분위수
q2 = np.percentile(data, 50)  # 2사분위수
q3 = np.percentile(data, 75)  # 3사분위수

print(f"Q1: {q1}, Q2: {q2}, Q3: {q3}")

df = df[(df['SMILES'].str.len() >= 37) & (df['SMILES'].str.len() <= 56)]
# smiles의 길이가 1 ~ 1423까지 다양하기에 학습 데이터 전체를 다 사용하게 된다면 
# 나중에 transformer 학습 시 'PAD'를 너무 많이 채워야 하기 때문에 조기 제거 수행 

Q1: 37.0, Q2: 46.0, Q3: 56.0


(805417, 3)

In [4]:
df = df.reset_index()
df.head()

,level_0,index,SMILES,ID
0,201120,211170,CN1CCN(CC1)C1=Nc2cc(Cl)ccc2Nc2ccccc12,CHEMBL42
1,210950,221680,COc1ccc2nccc(C(O)C3CC4CCN3CC4C=C)c2c1,CHEMBL97
2,212475,223637,NS(=O)(=O)c1cc(Cl)c(Cl)c(c1)S(N)(=O)=O,CHEMBL17
3,212923,224086,CC(Cc1ccc(O)c(O)c1)C(C)Cc1ccc(O)c(O)c1,CHEMBL52
4,212946,224109,CCCCC1C(=O)N(N(C1=O)c1ccccc1)c1ccccc1,CHEMBL101


In [ ]:
# 화학적으로 이상한게 존재함 그래서 그런 건 삭제
from rdkit import Chem

drop_idx = [] # 화학적으로 유효하지 않아서 버릴 dataframe의 인덱스

for idx, smiles in enumerate(df['SMILES']):
	mol = Chem.MolFromSmiles(smiles) # 화학 분자식을 분자 객체로 변환
	if mol is None: # 만약 화학적으로 유효하지 않으면 
		drop_idx.append(idx) # drop하기 위해 인덱스를 drop_idx에 추가함

df = df.drop(drop_idx)

[08:40:28] Explicit valence for atom # 17 O, 3, is greater than permitted
[08:40:30] Explicit valence for atom # 19 O, 3, is greater than permitted
[08:40:30] Explicit valence for atom # 19 O, 3, is greater than permitted
[08:40:36] Explicit valence for atom # 17 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 15 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 16 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 16 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 16 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 17 O, 3, is greater than permitted
[08:40:37] Explicit valence for atom # 16 O, 3, is greater than permitted
[08:40:38] Explicit valence for atom # 13 O, 3, is greater than permitted
[08:40:39] Explicit valence for atom # 17 O, 3, is greater than permitted
[08:40:40] Explicit valence for atom # 3 O, 3, is greater than permitted
[08:40:40] Explicit valence for atom # 

In [7]:
df = df.drop(['index','level_0'],axis=1).reset_index()
df.head()

,index,SMILES,ID
0,0,CN1CCN(CC1)C1=Nc2cc(Cl)ccc2Nc2ccccc12,CHEMBL42
1,1,COc1ccc2nccc(C(O)C3CC4CCN3CC4C=C)c2c1,CHEMBL97
2,2,NS(=O)(=O)c1cc(Cl)c(Cl)c(c1)S(N)(=O)=O,CHEMBL17
3,3,CC(Cc1ccc(O)c(O)c1)C(C)Cc1ccc(O)c(O)c1,CHEMBL52
4,4,CCCCC1C(=O)N(N(C1=O)c1ccccc1)c1ccccc1,CHEMBL101


In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

scaffold_list = []

# SMILES 예시
for smiles in df['SMILES']:
	mol = Chem.MolFromSmiles(smiles)
	scaffold = MurckoScaffold.GetScaffoldForMol(mol) 
    # 분자식에서 scaffold 추출하기
    # = scaffold를 기반으로 새로운 분자를 생성하는 모델을 만들기 위해 X값을 만드는 것
	scaffold_list.append(Chem.MolToSmiles(scaffold)) 
 
df['scaffold'] = scaffold_list

In [ ]:
import numpy as np

print(df['scaffold'].isnull().sum()) 
df = df[~df['scaffold'].isnull()] 
# scaffold가 추출된 행만 사용하겠다는 의미
# null값 개수가 0이라 사실 전부 유의미한 데이터라고 할 수 있음

0


In [ ]:
df = df[df['scaffold'] != df['SMILES']]
df = df.reset_index()
# 드물게 smiles와 scaffold가 같은 경우를 대비하여 해당 경우의 데이터를 제외함

In [ ]:
df.head()

,level_0,index,SMILES,ID,scaffold
0,0,0,CN1CCN(CC1)C1=Nc2cc(Cl)ccc2Nc2ccccc12,CHEMBL42,c1ccc2c(c1)N=C(N1CCNCC1)c1ccccc1N2
1,1,1,COc1ccc2nccc(C(O)C3CC4CCN3CC4C=C)c2c1,CHEMBL97,c1ccc2c(CC3CC4CCN3CC4)ccnc2c1
2,2,2,NS(=O)(=O)c1cc(Cl)c(Cl)c(c1)S(N)(=O)=O,CHEMBL17,c1ccccc1
3,3,3,CC(Cc1ccc(O)c(O)c1)C(C)Cc1ccc(O)c(O)c1,CHEMBL52,c1ccc(CCCCc2ccccc2)cc1
4,4,4,CCCCC1C(=O)N(N(C1=O)c1ccccc1)c1ccccc1,CHEMBL101,O=C1CC(=O)N(c2ccccc2)N1c1ccccc1


In [ ]:
from rdkit import Chem

canonical_smiles_list = []

for s in df['SMILES']:
    can_smi = Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True)  
    # Canonical SMILES 변환
    # 어떤 원자를 시작 지점으로 하는지, 고리 구조를 어떻게 표현할 지 등에 따라서 하나의 분자일지라도 
    # 그것을 표현하는 SMILES 문자열은 여러 개가 있을 수 있다
    # -> ai의 혼동을 막고자 표준화된 형태로 smiles를 변환함
    # = 서로 다른 smiles를 하나의 형태로 통일 가능 
    canonical_smiles_list.append(can_smi)
    
df['cano_smiles'] = canonical_smiles_list

In [ ]:
cano_scaffold = []

for s in df['scaffold']:
	try:
		can_smi = Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True)
		# 마찬가지로 scaffold도 표준화된 형식으로 변환
		cano_scaffold.append(can_smi)
	except:
		cano_scaffold.append('유효하지 않음')

df['cano_scaffold'] = cano_scaffold

[09:02:28] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10 11 12 13 14
[09:03:13] Can't kekulize mol.  Unkekulized atoms: 2 6 7 8 9 10 11
[09:03:21] Can't kekulize mol.  Unkekulized atoms: 2 3 4 11 12 13 14 15 16
[09:03:32] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 8 9 10
[09:03:38] Can't kekulize mol.  Unkekulized atoms: 2 3 4 11 12 13 14 15 16
[09:03:40] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12
[09:04:00] Can't kekulize mol.  Unkekulized atoms: 2 12 13 14 15 16 17
[09:04:15] Can't kekulize mol.  Unkekulized atoms: 2 6 7 8 9 10 11
[09:04:23] Can't kekulize mol.  Unkekulized atoms: 2 12 13 14 15 16 17
[09:04:46] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 8 9 10 11 12
[09:06:05] Can't kekulize mol.  Unkekulized atoms: 0 1 8 9 10 11 12 13 14


In [ ]:
df = df[df['cano_scaffold'] != '유효하지 않음']
# scaffold를 표준화 형태로 바꾸는 걸 실패한 경우는 데이터에서 제외함

In [ ]:
df = df[['cano_smiles','cano_scaffold']]
df = df.drop_duplicates()
df = df.reset_index()
df.to_csv('transformer_train_data.csv',index=False) # (799120, 3)
# 약 80만개의 데이터 확보